# Phase 6: VLM Refinement Stage

**Authors:** Mahesh & Furaha  
**Model:** Gemma 4 (Google DeepMind, open-source)  
**Purpose:** When the CV pipeline confidence drops below threshold, the VLM steps in to refine food identification, portion estimation, and nutrition data.

## Pipeline Flow
```
Image → YOLOv8 (detect) → EfficientNet-B0 (classify) → Portion → Nutrition
                                                                    ↓
                                              Confidence < 0.70? → VLM Refinement (Gemma 4)
                                                                    ↓
                                              Corrected JSON output
```

In [ ]:
import json
import base64
import time
import os
from getpass import getpass

print("Phase 6: VLM Refinement Setup")
print("=" * 40)

## 1. Install Dependencies

We use the Google Genai SDK (free) to call Gemma 4.

In [ ]:
!pip install -q google-genai pydantic Pillow

## 2. Set Up API Key

Get your **free** API key from: https://aistudio.google.com/apikey

In [ ]:
from google import genai
from google.genai import types

# Enter your API key when prompted
API_KEY = getpass("Enter your Google AI Studio API key: ")
client = genai.Client(api_key=API_KEY)

print("Client initialized!")
print(f"Model: gemma-4-27b-it")

## 3. Define JSON Schemas (Pydantic)

These schemas define exactly what goes in and out of the VLM.

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum

class RefinementAction(str, Enum):
    CONFIRMED = "confirmed"
    CORRECTED = "corrected"
    UNKNOWN = "unknown"

class RefinementReason(str, Enum):
    LOW_CONFIDENCE = "low_confidence"
    MULTIPLE_SIMILAR = "multiple_similar_predictions"
    NO_FOOD_DETECTED = "no_food_detected"

class CVNutrition(BaseModel):
    calories_kcal: float
    protein_g: float
    fat_g: float
    carbs_g: float
    fiber_g: float

class CVTopPrediction(BaseModel):
    label: str
    confidence: float = Field(ge=0.0, le=1.0)

class CVItem(BaseModel):
    item_id: int
    food_name: str
    display_name: str
    classification_confidence: float
    detection_confidence: Optional[float] = None
    top3_predictions: list[CVTopPrediction]
    estimated_grams: float
    portion_method: str
    bbox: list[int]
    nutrition_per_100g: CVNutrition
    nutrition_total: CVNutrition

class CVOutput(BaseModel):
    image_id: str
    status: str = "success"
    average_confidence: float
    plate_detected: bool
    items: list[CVItem]
    totals: CVNutrition

class VLMRequest(BaseModel):
    image_base64: str
    reason: RefinementReason
    trigger_threshold: float = 0.70
    cv_output: CVOutput

# Output schemas
class OriginalResult(BaseModel):
    food_name: str
    classification_confidence: float

class RefinedResult(BaseModel):
    food_name: str
    display_name: str
    vlm_confidence: float
    food_description: Optional[str] = None

class RefinedPortion(BaseModel):
    estimated_grams: float
    portion_method: str = "vlm_visual_estimate"
    vlm_confidence: float

class RefinedItem(BaseModel):
    item_id: int
    action: RefinementAction
    original: OriginalResult
    refined: RefinedResult
    portion: RefinedPortion
    nutrition_per_100g: CVNutrition
    nutrition_total: CVNutrition

class VLMResponse(BaseModel):
    image_id: str
    refinement_status: str = "completed"
    vlm_model: str
    confidence_threshold_used: float
    items: list[RefinedItem]
    totals: CVNutrition
    notes: list[str] = []
    processing_time_ms: int

print("Schemas defined:")
print(f"  Input:  VLMRequest ({len(VLMRequest.model_fields)} fields)")
print(f"  Output: VLMResponse ({len(VLMResponse.model_fields)} fields)")

## 4. Define VLM Prompts

In [ ]:
SYSTEM_PROMPT = """You are a food identification and nutrition analysis expert.
Your task is to review food images that a CV pipeline has already analyzed.

The CV pipeline uses YOLOv8 for detection and EfficientNet-B0 for classification
(Food-101 dataset, 101 classes). When its confidence is low, you step in to
refine the results.

You will receive:
1. An image of a meal/food
2. The CV pipeline's JSON output (may contain errors)

Your job:
- IDENTIFY each food item in the image (correct the CV if wrong)
- ESTIMATE portion size in grams for each item
- Provide nutritional data per 100g for each item

Rules:
- Use the Food-101 class names where possible (underscores, lowercase)
- Be honest about uncertainty
- Recalculate nutrition_total = (estimated_grams / 100) * nutrition_per_100g

You MUST respond with valid JSON matching the expected schema."""


def build_user_prompt(reason, threshold, avg_confidence, cv_output_json):
    return f"""I need you to refine the food analysis for this image.

## Reason for refinement:
- **Trigger**: {reason}
- **Confidence threshold**: {threshold}
- **CV average confidence**: {avg_confidence}

## CV Pipeline Output:
```json
{cv_output_json}
```

For each food item the CV detected, respond with JSON:
{{
  "items": [
    {{
      "item_id": <same>,
      "action": "confirmed" | "corrected" | "unknown",
      "original": {{"food_name": "<cv>", "classification_confidence": <val>}},
      "refined": {{"food_name": "<yours>", "display_name": "<readable>", "vlm_confidence": <0-1>, "food_description": "<desc>"}},
      "portion": {{"estimated_grams": <val>, "portion_method": "vlm_visual_estimate", "vlm_confidence": <0-1>}},
      "nutrition_per_100g": {{"calories_kcal": <val>, "protein_g": <val>, "fat_g": <val>, "carbs_g": <val>, "fiber_g": <val>}},
      "nutrition_total": {{"calories_kcal": <val>, "protein_g": <val>, "fat_g": <val>, "carbs_g": <val>, "fiber_g": <val>}}
    }}
  ]
}}

Respond with ONLY the JSON."""

print(f"System prompt: {len(SYSTEM_PROMPT)} chars")
print(f"User prompt template ready")

## 5. VLM Client — Call Gemma 4

In [ ]:
def call_gemma4(image_base64, prompt, model="gemma-4-27b-it"):
    """Call Gemma 4 via Google AI Studio."""
    image_bytes = base64.b64decode(image_base64)
    
    start = time.time()
    response = client.models.generate_content(
        model=model,
        contents=[
            types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg"),
            prompt,
        ],
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=1.0,
            top_p=0.95,
            top_k=64,
        ),
    )
    elapsed = int((time.time() - start) * 1000)
    
    return response.text, elapsed


def extract_json(text):
    """Extract JSON from model response."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    if "```json" in text:
        s = text.index("```json") + 7
        e = text.index("```", s)
        return json.loads(text[s:e].strip())
    elif "```" in text:
        s = text.index("```") + 3
        e = text.index("```", s)
        return json.loads(text[s:e].strip())
    elif "{" in text:
        s = text.index("{")
        e = text.rindex("}") + 1
        return json.loads(text[s:e])
    raise ValueError(f"No JSON found in response")

print("Gemma 4 client ready")

## 6. End-to-End Demo — Single Image

In [ ]:
from PIL import Image
import io

# Upload a food image
from google.colab import files
uploaded = files.upload()
image_name = list(uploaded.keys())[0]

# Encode image
img = Image.open(io.BytesIO(uploaded[image_name]))
img = img.convert('RGB')
buf = io.BytesIO()
img.save(buf, format='JPEG')
img_b64 = base64.b64encode(buf.getvalue()).decode()

print(f"Image loaded: {image_name} ({len(img_b64)} chars)")
img.show()

In [ ]:
# Create mock CV pipeline output
cv_output = CVOutput(
    image_id="demo-001",
    average_confidence=0.55,
    plate_detected=True,
    items=[CVItem(
        item_id=1,
        food_name="spaghetti_bolognese",
        display_name="Spaghetti Bolognese",
        classification_confidence=0.55,
        detection_confidence=0.80,
        top3_predictions=[
            CVTopPrediction(label="spaghetti_bolognese", confidence=0.55),
            CVTopPrediction(label="ramen", confidence=0.20),
            CVTopPrediction(label="pho", confidence=0.10),
        ],
        estimated_grams=250,
        portion_method="plate_reference",
        bbox=[100, 50, 400, 350],
        nutrition_per_100g=CVNutrition(
            calories_kcal=132, protein_g=5.8,
            fat_g=4.5, carbs_g=18.1, fiber_g=1.4
        ),
        nutrition_total=CVNutrition(
            calories_kcal=330, protein_g=14.5,
            fat_g=11.3, carbs_g=45.3, fiber_g=3.5
        ),
    )],
    totals=CVNutrition(
        calories_kcal=330, protein_g=14.5,
        fat_g=11.3, carbs_g=45.3, fiber_g=3.5
    ),
)

# Build request
request = VLMRequest(
    image_base64=img_b64,
    reason=RefinementReason.LOW_CONFIDENCE,
    trigger_threshold=0.70,
    cv_output=cv_output,
)

# Build prompt
user_prompt = build_user_prompt(
    reason=request.reason.value,
    threshold=request.trigger_threshold,
    avg_confidence=request.cv_output.average_confidence,
    cv_output_json=request.cv_output.model_dump_json(),
)

print(f"CV prediction: {cv_output.items[0].food_name} ({cv_output.average_confidence:.0%})")
print("Calling Gemma 4...")

raw_response, elapsed = call_gemma4(img_b64, user_prompt)
parsed = extract_json(raw_response)

print(f"\nResponse ({elapsed}ms):")
print(json.dumps(parsed, indent=2))

## 7. Validation — Parse into Pydantic Schema

In [ ]:
# Validate VLM response against our schema
items_data = parsed.get("items", parsed)

print("=" * 50)
print("VLM Refinement Result")
print("=" * 50)

for item in items_data:
    action = item.get("action", "unknown")
    orig = item.get("original", {}).get("food_name", "?")
    refined = item.get("refined", {}).get("food_name", "?")
    conf = item.get("refined", {}).get("vlm_confidence", 0)
    grams = item.get("portion", {}).get("estimated_grams", 0)
    
    icon = "CONFIRMED" if action == "confirmed" else "CORRECTED"
    print(f"\n  Action:    {icon}")
    print(f"  CV said:   {orig}")
    print(f"  VLM says:  {refined} ({conf:.0%} confidence)")
    print(f"  Portion:   {grams}g")
    
    n = item.get("nutrition_per_100g", {})
    print(f"  Calories:  {n.get('calories_kcal', 0)} kcal/100g")
    print(f"  Protein:   {n.get('protein_g', 0)}g/100g")
    print(f"  Fat:       {n.get('fat_g', 0)}g/100g")
    print(f"  Carbs:     {n.get('carbs_g', 0)}g/100g")
    
    desc = item.get("refined", {}).get("food_description", "")
    if desc:
        print(f"  Note:      {desc}")

## 8. Multi-Image Evaluation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Upload multiple food images
print("Upload multiple food images for evaluation...")
uploaded = files.upload()
image_files = list(uploaded.keys())
print(f"\nUploaded {len(image_files)} images")

# Show uploaded images
fig, axes = plt.subplots(1, len(image_files), figsize=(4*len(image_files), 4))
if len(image_files) == 1:
    axes = [axes]
for ax, name in zip(axes, image_files):
    img = Image.open(io.BytesIO(uploaded[name]))
    ax.imshow(img)
    ax.set_title(name, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Test cases with mock CV predictions
EVAL_CASES = [
    {"file": image_files[0], "cv_prediction": "hot_dog", "confidence": 0.35},
    # Add more as needed
]

# Extend if more images
cv_predictions = ["hot_dog", "ramen", "caesar_salad", "french_toast",
                  "frozen_yogurt", "pho", "red_velvet_cake", "filet_mignon"]
for i, name in enumerate(image_files[1:], 1):
    pred = cv_predictions[(i-1) % len(cv_predictions)]
    EVAL_CASES.append({"file": name, "cv_prediction": pred, "confidence": 0.35 + (i*0.03)})

results = []

for i, case in enumerate(EVAL_CASES):
    name = case["file"]
    print(f"\n[{i+1}/{len(EVAL_CASES)}] {name}")
    print(f"  CV says: {case['cv_prediction']} ({case['confidence']:.0%})")
    
    # Encode image
    img = Image.open(io.BytesIO(uploaded[name])).convert('RGB')
    buf = io.BytesIO()
    img.save(buf, format='JPEG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    
    # Simple prompt for speed
    prompt = (
        f"The CV pipeline predicted this food is "
        f"'{case['cv_prediction']}' with {case['confidence']:.0%} confidence. "
        f"Look at the image and identify the correct food. "
        f"Respond with JSON: "
        f"{{\"food_name\": \"name\", \"display_name\": \"Name\", "
        f"\"action\": \"confirmed or corrected\", "
        f"\"vlm_confidence\": 0.9, \"estimated_grams\": 250}}"
    )
    
    print("  Calling Gemma 4...", end="", flush=True)
    try:
        raw, elapsed = call_gemma4(img_b64, prompt)
        parsed = extract_json(raw)
        vlm_food = parsed.get("food_name", "?")
        vlm_conf = parsed.get("vlm_confidence", 0)
        action = parsed.get("action", "?")
        grams = parsed.get("estimated_grams", 0)
        print(f" {elapsed/1000:.1f}s")
        print(f"  VLM says: {vlm_food} ({vlm_conf:.0%}) [{action.upper()}]")
        results.append({"file": name, "cv": case["cv_prediction"],
                       "vlm": vlm_food, "confidence": vlm_conf,
                       "action": action, "grams": grams,
                       "time_ms": elapsed})
    except Exception as e:
        print(f" ERROR: {e}")
        results.append({"file": name, "cv": case["cv_prediction"],
                       "vlm": "error", "error": str(e)})

## 9. Evaluation Results

In [ ]:
# Summary table
print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(f"{'Image':<20} {'CV Predicted':<20} {'VLM Predicted':<20} {'Action':<12}")
print("-" * 72)
for r in results:
    print(f"{r['file']:<20} {r['cv']:<20} {r.get('vlm','?'):<20} {r.get('action','?'):<12}")

# Stats
total = len(results)
corrected = sum(1 for r in results if r.get('action') == 'corrected')
confirmed = sum(1 for r in results if r.get('action') == 'confirmed')
avg_time = sum(r.get('time_ms', 0) for r in results) / max(total, 1)
avg_conf = sum(r.get('confidence', 0) for r in results) / max(total, 1)

print(f"\nTotal: {total} images")
print(f"Corrected: {corrected} | Confirmed: {confirmed}")
print(f"Avg VLM confidence: {avg_conf:.1%}")
print(f"Avg time: {avg_time/1000:.1f}s per image")

In [ ]:
# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Chart 1: Action distribution
actions = [r.get('action', 'unknown') for r in results]
from collections import Counter
action_counts = Counter(actions)
ax1.bar(action_counts.keys(), action_counts.values(), color=['#2ecc71', '#e74c3c', '#95a5a6'])
ax1.set_title('VLM Actions')
ax1.set_ylabel('Count')

# Chart 2: Confidence comparison
cv_confs = [0.35 + i*0.03 for i in range(len(results))]
vlm_confs = [r.get('confidence', 0) for r in results]
x = range(len(results))
ax2.bar([i - 0.15 for i in x], cv_confs, width=0.3, label='CV Pipeline', color='#e74c3c', alpha=0.7)
ax2.bar([i + 0.15 for i in x], vlm_confs, width=0.3, label='VLM (Gemma 4)', color='#2ecc71', alpha=0.7)
ax2.set_title('Confidence: CV vs VLM')
ax2.set_ylabel('Confidence')
ax2.legend()
ax2.set_xticks(x)
ax2.set_xticklabels([r['file'][:10] for r in results], rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig('vlm_evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved as vlm_evaluation_results.png")

## 10. Architecture Summary

### VLM Module Structure
```
vlm/
├── __init__.py      # Module header
├── schemas.py       # Pydantic models (VLMRequest, VLMResponse)
├── prompts.py       # System + user prompt templates
├── client.py        # Gemma 4 API client (Ollama + Google AI Studio)
├── refiner.py       # Orchestrator: validate → prompt → call → parse
├── example_input.json   # Sample CV pipeline output
└── example_output.json  # Sample VLM correction
```

### Key Design Decisions
| Decision | Choice | Reason |
|----------|--------|--------|
| VLM Model | Gemma 4 (e4b / 27b-it) | Open-source, multimodal, strong vision |
| Backend | Ollama (local) + Google AI Studio (cloud) | Free, no API key needed for local |
| Confidence Threshold | 0.70 | TBD with Luca's team |
| JSON Validation | Pydantic v2 | Type safety, auto-validation |
| Correction Actions | confirmed / corrected / unknown | Tracks VLM decisions |

### Evaluation Results (10 images, gemma4:e4b via Ollama, local)

| Metric | CV Pipeline | After VLM (Gemma 4) | Change |
|--------|------------|---------------------|--------|
| Exact Match | 30% (3/10) | 50% (5/10) | **+20%** |
| Exact + Partial | 40% (4/10) | 70% (7/10) | **+30%** |
| Wrong | 60% (6/10) | 30% (3/10) | **-30%** |

#### Per-Image Results
| Image | CV Said | VLM Said | Actual | Match |
|-------|---------|----------|--------|-------|
| pizza | pizza | pizza | pizza | EXACT |
| burger | hot_dog | burger | hamburger | PARTIAL |
| sushi | sushi | Sushi Rolls | sushi | PARTIAL |
| pasta | ramen | pasta | spaghetti_bolognese | WRONG |
| salad | caesar_salad | Salad Bowl | greek_salad | WRONG |
| pancakes | french_toast | pancakes | pancakes | EXACT |
| ice cream | frozen_yogurt | ice cream | ice_cream | EXACT |
| steak | filet_mignon | steak | steak | EXACT |
| ramen | pho | Shrimp Noodle Soup | ramen | WRONG |
| chocolate cake | red_velvet_cake | chocolate_cake | chocolate_cake | EXACT |

- **Avg VLM confidence:** 95% (vs CV avg 41%)
- **Avg inference time:** 146s per image (NVIDIA T500, CPU/GPU split)
- **Cost:** $0.00 (fully local, open-source)
- **Corrections made:** 8/10 | **Corrections correct:** 5/8 (63%)